In [1]:
"""
Reproduces Fig. 5 through Fig. 10 from Ratnakar (2026), Int. J. Hydrogen
Energy 245, 155702 -- the residual/real-state part (Section 3.3), which
requires solving vapor-liquid equilibrium (VLE) from the PR-78 EOS.

Per the paper's own scope (Sec. 3.3: "For brevity of the demonstration of
the EOS model, only pH2 is considered in this section"), all Fig.6-10
reproductions here use pH2 only.

VLE is solved by the standard successive-substitution method: at fixed T,
adjust P until the fugacity coefficients of the liquid-branch and
vapor-branch roots of the SAME cubic are equal (phi_liq = phi_vap). This
is not shown explicitly in the paper (it references standard textbooks
for this), but it is the standard method for any cubic EOS.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from h2_thermo_final import (
    R, EOS_PARAMS, eos_ac_b, a_of_T, molar_volume_PR, real_properties,
    sonic_velocity, joule_thomson,
)

OUTDIR = "figures"
Mw_H2 = 2.01588e-3  # kg/mol
SPECIES = "p"  # paper only demonstrates the residual/real-state part for pH2

# =======================================================================
# VLE: fugacity coefficient (standard PR-78 result) and saturation P(T)
# =======================================================================
def fugacity_coeff(T, P, species, phase):
    ac, b, m, Tc, Cpen = eos_ac_b(species)
    a = a_of_T(T, species)
    V_corr, Z = molar_volume_PR(T, P, species, phase)
    A = a * P / (R*T)**2
    B = b * P / (R*T)
    sqrt2 = np.sqrt(2)
    ln_phi = ((Z - 1) - np.log(Z - B)
              - (A/(2*sqrt2*B)) * np.log((Z + (1+sqrt2)*B) / (Z + (1-sqrt2)*B)))
    return np.exp(ln_phi)


def saturation_pressure(T, species, P_guess=None, tol=1e-10, max_iter=300):
    """Successive substitution: P_(n+1) = P_n * phi_liq/phi_vap, until phi_liq=phi_vap."""
    Pc = EOS_PARAMS[species]["Pc"]
    Tc = EOS_PARAMS[species]["Tc"]
    omega = EOS_PARAMS[species]["omega"]
    if P_guess is None:
        Tr = T / Tc
        P_guess = Pc * 10**(7.0/3.0*(1+omega)*(1-1/Tr))  # Lee-Kesler-type initial guess
    P = P_guess
    for _ in range(max_iter):
        phi_l = fugacity_coeff(T, P, species, "liquid")
        phi_v = fugacity_coeff(T, P, species, "vapor")
        ratio = phi_l / phi_v
        P_new = P * ratio
        if abs(ratio - 1) < tol:
            return P_new
        P = P_new
    return P


# =======================================================================
# FIGURE 5a: P-T phase diagram (saturation curve from PR-78 VLE)
# =======================================================================
Tc = EOS_PARAMS[SPECIES]["Tc"]
Pc = EOS_PARAMS[SPECIES]["Pc"]
Tt = EOS_PARAMS[SPECIES]["Tt"]
Pt = EOS_PARAMS[SPECIES]["Pt"]

T_sat_grid = np.linspace(Tt, Tc - 0.02, 150)
P_sat_grid = []
P_prev = None
for T in T_sat_grid:
    P_prev = saturation_pressure(T, SPECIES, P_guess=P_prev)
    P_sat_grid.append(P_prev)
P_sat_grid = np.array(P_sat_grid)

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.plot(T_sat_grid, P_sat_grid/1e5, color="tab:blue", label="Vapor-liquid (PR-78 EOS, this work)")
ax.scatter([Tc], [Pc/1e5], color="red", zorder=5, label=f"Critical point ({Tc:.2f}, {Pc/1e5:.2f})")
ax.scatter([Tt], [Pt/1e5], color="green", zorder=5, label=f"Triple point ({Tt:.2f}, {Pt/1e5:.4f})")
ax.set_yscale("log")
ax.set_xlabel("T, K")
ax.set_ylabel("P, bar")
ax.set_title("pH2 P-T phase diagram (reproduces paper's Fig. 5a)\n"
              "(solid-liquid/solid-vapor boundaries not modeled -- PR-EOS has no solid phase)",
              fontsize=10)
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(f"{OUTDIR}/fig5a_phase_diagram_PT.png", dpi=150)
plt.close(fig)
print("Saved fig5a_phase_diagram_PT.png")


# =======================================================================
# FIGURE 5b: P-V envelope + isotherms
# =======================================================================
def P_of_V_eos(V_eos, T, species):
    ac, b, m, Tc_, Cpen = eos_ac_b(species)
    a = a_of_T(T, species)
    alpha1, alpha2 = 1+np.sqrt(2), 1-np.sqrt(2)
    return R*T/(V_eos-b) - a/((V_eos+alpha1*b)*(V_eos+alpha2*b))

ac, b, m, _, Cpen = eos_ac_b(SPECIES)

fig, ax = plt.subplots(figsize=(7.5, 6))
# saturation envelope (liquid branch + vapor branch), for shading the dome
V_liq_env, V_vap_env = [], []
for T, Psat in zip(T_sat_grid, P_sat_grid):
    Vl, _ = molar_volume_PR(T, Psat, SPECIES, phase="liquid")
    Vv, _ = molar_volume_PR(T, Psat, SPECIES, phase="vapor")
    V_liq_env.append(Vl*1e6)
    V_vap_env.append(Vv*1e6)
ax.plot(V_liq_env, P_sat_grid/1e5, color="tab:blue", label="Liquid branch (saturation)")
ax.plot(V_vap_env, P_sat_grid/1e5, color="tab:green", label="Vapor branch (saturation)")

# isotherms
for T_iso in [20, 25, 30, 32, 35, 50, 100, 200, 300]:
    V_min = b + Cpen + 1e-7
    V_max = 5e-2
    V_grid_eos = np.geomspace(max(V_min, b*1.001), V_max, 300)
    P_iso = P_of_V_eos(V_grid_eos, T_iso, SPECIES)
    V_grid_corr = (V_grid_eos - Cpen) * 1e6
    mask = (P_iso > 0) & (P_iso < 200e5)
    ax.plot(V_grid_corr[mask], P_iso[mask]/1e5, lw=0.8, color="gray", alpha=0.7)
    # label near a representative point
    idx = np.argmin(np.abs(P_iso[mask]/1e5 - 5)) if np.any(mask) else None

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(15, 2e4)
ax.set_ylim(0.05, 200)
ax.set_xlabel(r"V, cm$^3$/mol")
ax.set_ylabel("P, bar")
ax.set_title("pH2 P-V phase envelope and isotherms (reproduces paper's Fig. 5b)\n"
              "(gray lines = isotherms at 20,25,30,32,35,50,100,200,300K)")
ax.legend(fontsize=8)
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(f"{OUTDIR}/fig5b_phase_envelope_PV.png", dpi=150)
plt.close(fig)
print("Saved fig5b_phase_envelope_PV.png")


# =======================================================================
# Helper: Tsat(P) by inverting saturation_pressure(T) -> P
# =======================================================================
def T_saturation(P_target, species, T_bracket=None):
    Tc_ = EOS_PARAMS[species]["Tc"]
    Tt_ = EOS_PARAMS[species]["Tt"]
    if P_target >= EOS_PARAMS[species]["Pc"]:
        return None  # supercritical pressure, no VLE
    f = lambda T: saturation_pressure(T, species) - P_target
    lo, hi = Tt_, Tc_ - 0.01
    try:
        return brentq(f, lo, hi, xtol=1e-6)
    except ValueError:
        return None


# =======================================================================
# FIGURES 6-10: density, Cp, Cv, sonic velocity, Joule-Thomson vs T,
# at P = 0.1, 1, 10, 20 bar (pH2 only, matching paper's Figs.6-10)
# =======================================================================
pressures_bar = [0.1, 1, 10, 20]
T_full = np.linspace(15, 600, 250)

def scan_property(P_bar, prop_func):
    """Return T, values for one property function across T_full, switching
    phase at the saturation temperature for this pressure (if any)."""
    P = P_bar * 1e5
    Tsat = T_saturation(P, SPECIES)
    T_out, V_out = [], []
    for T in T_full:
        if Tsat is not None and T < Tsat:
            phase = "liquid"
        else:
            phase = "vapor"
        try:
            val = prop_func(T, P, phase)
        except Exception:
            val = np.nan
        T_out.append(T)
        V_out.append(val)
    return np.array(T_out), np.array(V_out), Tsat


def get_density(T, P, phase):
    props = real_properties(T, P, SPECIES, phase=phase)
    return Mw_H2 / props["V_corr"]

def get_Cp(T, P, phase):
    return real_properties(T, P, SPECIES, phase=phase)["Cp"]

def get_Cv(T, P, phase):
    return real_properties(T, P, SPECIES, phase=phase)["Cv"]

def get_sonic(T, P, phase):
    return sonic_velocity(T, P, SPECIES, phase=phase)

def get_JT(T, P, phase):
    return joule_thomson(T, P, SPECIES, phase=phase) * 1e5  # K/Pa -> K/bar


def make_4panel_figure(prop_func, ylabel, title, fname, logy=False):
    fig, axes = plt.subplots(2, 2, figsize=(11, 9))
    for ax, P_bar, tag in zip(axes.flat, pressures_bar, ["(a)", "(b)", "(c)", "(d)"]):
        T_out, V_out, Tsat = scan_property(P_bar, prop_func)
        ax.plot(T_out, V_out, color="tab:orange", lw=1.5)
        if Tsat is not None:
            ax.axvline(Tsat, color="gray", ls=":", lw=1)
        ax.set_xlabel("T, K")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{tag} P = {P_bar} bar", loc="left", fontweight="bold")
        if logy:
            ax.set_yscale("log")
        ax.grid(alpha=0.3)
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(f"{OUTDIR}/{fname}", dpi=150)
    plt.close(fig)
    print(f"Saved {fname}")


make_4panel_figure(get_density, r"$\rho$, kg/m$^3$",
                    "pH2 density vs T (reproduces paper's Fig. 6)",
                    "fig6_density.png")

make_4panel_figure(get_Cp, r"$C_P/R$",
                    "pH2 isobaric heat capacity vs T (reproduces paper's Fig. 7)",
                    "fig7_Cp_real.png")
# Cp/R needs dividing by R -- wrap:
def get_Cp_over_R(T, P, phase):
    return get_Cp(T, P, phase) / R
make_4panel_figure(get_Cp_over_R, r"$C_P/R$",
                    "pH2 isobaric heat capacity vs T (reproduces paper's Fig. 7)",
                    "fig7_Cp_real.png")

def get_Cv_over_R(T, P, phase):
    return get_Cv(T, P, phase) / R
make_4panel_figure(get_Cv_over_R, r"$C_V/R$",
                    "pH2 isochoric heat capacity vs T (reproduces paper's Fig. 8)",
                    "fig8_Cv_real.png")

make_4panel_figure(get_sonic, "sonic velocity, m/s",
                    "pH2 sonic velocity vs T (reproduces paper's Fig. 9)",
                    "fig9_sonic_velocity.png")

make_4panel_figure(get_JT, r"$\mu_{JT}$, K/bar",
                    "pH2 Joule-Thomson coefficient vs T (reproduces paper's Fig. 10)",
                    "fig10_joule_thomson.png")

print("\nAll figures done.")

Saved fig5a_phase_diagram_PT.png
Saved fig5b_phase_envelope_PV.png
Saved fig6_density.png
Saved fig7_Cp_real.png
Saved fig7_Cp_real.png
Saved fig8_Cv_real.png
Saved fig9_sonic_velocity.png
Saved fig10_joule_thomson.png

All figures done.
